<a href="https://colab.research.google.com/github/Mariano-rr/ThinkPythonAssignments/blob/main/Week15.5/NutritionToolkit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install pandas
import pandas as pd
import json
import os
from datetime import datetime, timedelta

class NutritionToolkit:
    def __init__(self):
        self.macro_file = "macro_log.csv"
        self.fridge_file = "fridge_log.csv"
        self.recipe_file = "recipes.json"
        self.goals = {'p': 120, 'c': 200, 'f': 70}
        self._init_files()

    def _init_files(self):
        """Initializes storage and prevents corruption from empty files."""
        try:
            if not os.path.exists(self.macro_file) or os.path.getsize(self.macro_file) == 0:
                pd.DataFrame(columns=['date', 'food', 'p_grams', 'c_grams', 'f_grams', 'calories']).to_csv(self.macro_file, index=False)
            if not os.path.exists(self.fridge_file) or os.path.getsize(self.fridge_file) == 0:
                pd.DataFrame(columns=['item', 'cooked_date', 'expiry_date']).to_csv(self.fridge_file, index=False)
            if not os.path.exists(self.recipe_file) or os.path.getsize(self.recipe_file) == 0:
                with open(self.recipe_file, 'w') as f:
                    json.dump({}, f)
        except PermissionError:
            print("\n[!] ACCESS ERROR: Close your CSV files (like Excel) before running!")

    def log_meal(self, name, p, c, f):
        """Logs nutritional data with explicit units (g)."""
        p, c, f = max(0, p), max(0, c), max(0, f)
        calories = (p * 4) + (c * 4) + (f * 9)
        new_row = pd.DataFrame([[str(datetime.now().date()), name.strip(), p, c, f, calories]],
                               columns=['date', 'food', 'p_grams', 'c_grams', 'f_grams', 'calories'])
        new_row.to_csv(self.macro_file, mode='a', header=False, index=False)
        print(f"\n[✓] Logged: {name} | {p}g Protein, {c}g Carbs, {f}g Fat | Total: {calories:.1f} kcal")

    def add_leftover(self, item_name):
        """Logs fridge items with automatic 4-day safety window."""
        if not item_name.strip(): return
        expires = datetime.now().date() + timedelta(days=4)
        new_item = pd.DataFrame([[item_name.strip(), str(datetime.now().date()), str(expires)]],
                                columns=['item', 'cooked_date', 'expiry_date'])
        new_item.to_csv(self.fridge_file, mode='a', header=False, index=False)
        print(f"\n[!] Added: {item_name}. Eat by {expires}.")

    def check_fridge(self):
        """Displays status by filtering for unexpired items."""
        if not os.path.exists(self.fridge_file): return
        df = pd.read_csv(self.fridge_file)
        if df.empty: print("\nFridge is empty!"); return
        df['expiry_date'] = pd.to_datetime(df['expiry_date']).dt.date
        safe_items = df[df['expiry_date'] >= datetime.now().date()]
        print(f"\n--- FRIDGE STATUS ---")
        print(safe_items[['item', 'expiry_date']].to_string(index=False) if not safe_items.empty else "No safe items.")

    def save_recipe(self, name, servings, ingredients):
        """Saves recipes case-insensitively with nested ingredient data."""
        with open(self.recipe_file, 'r') as f:
            recipes = json.load(f)
        # Store as lowercase key for easier lookup later
        recipes[name.lower().strip()] = {"display_name": name.strip(), "servings": servings, "ingredients": ingredients}
        with open(self.recipe_file, 'w') as f:
            json.dump(recipes, f, indent=4)
        print(f"\n[✓] Saved recipe '{name}'.")

    def list_recipes(self):
        """Displays saved library so users know what they can scale."""
        with open(self.recipe_file, 'r') as f:
            recipes = json.load(f)
        if not recipes:
            print("\n[!] No recipes saved yet.")
            return False
        print("\n--- YOUR SAVED RECIPES ---")
        for r in recipes.values():
            print(f"- {r['display_name']} ({r['servings']} servings)")
        return True

    def scale_and_export(self, recipe_name, target_servings):
        """Scales quantities, calculates total cost, and exports to TXT."""
        with open(self.recipe_file, 'r') as f:
            recipes = json.load(f)

        search_key = recipe_name.lower().strip()
        if not search_key or search_key not in recipes:
            print(f"\n[X] Error: '{recipe_name}' not found.")
            return

        data = recipes[search_key]
        factor = target_servings / data['servings']
        total_cost = 0
        safe_fname = "".join(x for x in data['display_name'] if x.isalnum() or x in " -_").strip().lower()
        filename = f"shopping_list_{safe_fname}.txt"

        with open(filename, "w") as f:
            f.write(f"SHOPPING LIST: {data['display_name']} ({target_servings} Servings)\n" + "="*60 + "\n")
            f.write(f"{'Item':<18} | {'Qty Needed':>12} | {'Cost (Total)':>12}\n" + "-"*60 + "\n")
            for item, details in data['ingredients'].items():
                amt, unit_price = details
                scaled_amt = amt * factor
                cost = scaled_amt * unit_price
                total_cost += cost
                f.write(f"{item:<18} | {scaled_amt:>12.2f} | ${cost:>11.2f}\n")
            f.write("="*60 + f"\nTOTAL ESTIMATED RECIPE COST: ${total_cost:.2f}\n")
        print(f"\n[✓] List exported to: {filename}")

def safe_input(prompt, type_=float):
    """Recursively validates numeric input."""
    while True:
        try:
            val = type_(input(prompt))
            if val < 0: print("Error: Enter a positive number."); continue
            return val
        except ValueError: print("Invalid input. Please enter a number.")

def main():
    kit = NutritionToolkit()
    while True:
        print("\n--- NUTRITION & COST TOOLKIT ---")
        print("1: Log Meal | 2: Fridge | 3: Save Recipe | 4: Scale Recipe | 5: Exit")
        cmd = input("Select Option: ").strip()

        if cmd == "1":
            kit.log_meal(input("Food Name: "), safe_input("Protein (g): "), safe_input("Carbs (g): "), safe_input("Fat (g): "))
        elif cmd == "2":
            sub = input("(A)dd or (V)iew? ").lower().strip()
            if sub == 'a': kit.add_leftover(input("Item Name: "))
            else: kit.check_fridge()
        elif cmd == "3":
            name = input("Recipe Name: ")
            serv = safe_input("Base Servings: ", int)
            print("Format: Item:Amt:Price_Per_1_Unit (e.g., Chicken:500:0.02)")
            raw = input("List ingredients: ")
            ing = {}
            for entry in raw.split(','):
                p = entry.split(':')
                if len(p) == 3:
                    # Correctly mapping: name = index 0, amt = index 1, price = index 2
                    ing[p[0].strip()] = [float(p[1]), float(p[2])]
            if ing: kit.save_recipe(name, serv, ing)
            else: print("[X] Formatting Error: Use 'Name:Amt:Price'.")
        elif cmd == "4":
            if kit.list_recipes():
                kit.scale_and_export(input("\nName of recipe to scale: "), safe_input("Target Servings: ", int))
        elif cmd == "5":
            print("Exiting... Stay healthy!")
            break

if __name__ == "__main__":
    main()
